## Acertijo de Einstein

![alt text](Acertijo-Einstein.png "Title")

### Condiciones dadas (lo que sabemos del mundo)

1. El brasilero vive en la casa roja.
2. El venezolano tiene un perro como mascota.
3. El ecuatoriano toma té.
4. La peruana vive en la primera casa.
5. La colombiana ve "Westworld".
6. La casa verde está inmediatamente a la izquierda de la blanca
7. El dueño de la casa verde bebe café.
8. El propietario que ve "The Boys" cría pájaros.
9. El dueño de la casa amarilla ve "Dexter".
10. El hombre que vive en la casa del centro bebe leche.
11. El vecino que ve "El Patrón del Mal" vive al lado del que tiene un gato.
12. El hombre que tiene un caballo vive al lado del que ve "Dexter".
13. El propietario que ve "Breaking Bad" toma cerveza.
14. El vecino que ve "El Patrón del Mal" vive al lado del que toma agua.
15. La peruana vive al lado de la casa azul.


### Problema a resolver:

Queremos saber quién vive en cada casa, es decir, su nacionalidad, qué mascota tiene, qué serie ve, qué bebida bebe y qué color tiene la casa. 

Es decir, queremos llenar la siguiente tabla completamente de manera que se satisfagan _todas_ las condiciones de arriba:


|Id-Casa| Color | Nacionalidad | Mascota | Serie | Bebida |
|---|---|---|---|---|---|
| Casa 0 | ? | ? | ? | ? | ? |
| Casa 1 | ? | ? | ? | ? | ? |
| Casa 2 | ? | ? | ? | ? | ? |
| Casa 3 | ? | ? | ? | ? | ? |
| Casa 4 | ? | ? | ? | ? | ? |



## Intentos de solución con LLMs

In [2]:
LLM_PROMPT = """
Por favor ayúdame a resolver el siguiente acertijo y dame la respuesta en formato JSON, como una lista de diccionarios. 

En una calle hay cinco casas, cada una de un color distinto. 
En cada casa vive una persona de distinta nacionalidad. 
Cada dueño bebe un único tipo de bebida,
ve una sola Serie de TV y tiene una mascota diferente a sus vecinos.  

1. El brasilero vive en la casa roja.
2. El venezolano tiene un perro como mascota.
3. El ecuatoriano toma té.
4. La peruana vive en la primera casa.
5. La colombiana ve "Westworld".
6. La casa verde está inmediatamente a la izquierda de la blanca
7. El dueño de la casa verde bebe café.
8. El propietario que ve "The Boys" cría pájaros.
9. El dueño de la casa amarilla ve "Dexter".
10. El hombre que vive en la casa del centro bebe leche.
11. El vecino que ve "El Patrón del Mal" vive al lado del que tiene un gato.
12. El hombre que tiene un caballo vive al lado del que ve "Dexter".
13. El propietario que ve "Breaking Bad" toma cerveza.
14. El vecino que ve "El Patrón del Mal" vive al lado del que toma agua.
15. La peruana vive al lado de la casa azul.

Cada diccionario en tu respuesta en formato JSON debe tener exactamente las siguientes llaves y ninguna otra:  
"nacionalidad", "bebida_que_toma", "serie_que_ve", "color", "mascota".
"""

### Condiciones en Español

In [3]:
CONDICIONES_ESPANHOL = [ line.split(".")[1].strip() 
                         for line in 
                         """0. En una calle hay cinco casas.
                            1. El brasilero vive en la casa roja.
                            2. El venezolano tiene un perro como mascota.
                            3. El ecuatoriano toma té.
                            4. La peruana vive en la primera casa.
                            5. La colombiana ve "Westworld".
                            6. La casa verde está inmediatamente a la izquierda de la blanca
                            7. El dueño de la casa verde bebe café.
                            8. El propietario que ve "The Boys" cría pájaros.
                            9. El dueño de la casa amarilla ve "Dexter".
                            10. El hombre que vive en la casa del centro bebe leche.
                            11. El vecino que ve "El Patrón del Mal" vive al lado del que tiene un gato.
                            12. El hombre que tiene un caballo vive al lado del que ve "Dexter".
                            13. El propietario que ve "Breaking Bad" toma cerveza.
                            14. El vecino que ve "El Patrón del Mal" vive al lado del que toma agua.
                            15. La peruana vive al lado de la casa azul."""
                             .split("\n")
                       ]
CONDICIONES_ESPANHOL

['En una calle hay cinco casas',
 'El brasilero vive en la casa roja',
 'El venezolano tiene un perro como mascota',
 'El ecuatoriano toma té',
 'La peruana vive en la primera casa',
 'La colombiana ve "Westworld"',
 'La casa verde está inmediatamente a la izquierda de la blanca',
 'El dueño de la casa verde bebe café',
 'El propietario que ve "The Boys" cría pájaros',
 'El dueño de la casa amarilla ve "Dexter"',
 'El hombre que vive en la casa del centro bebe leche',
 'El vecino que ve "El Patrón del Mal" vive al lado del que tiene un gato',
 'El hombre que tiene un caballo vive al lado del que ve "Dexter"',
 'El propietario que ve "Breaking Bad" toma cerveza',
 'El vecino que ve "El Patrón del Mal" vive al lado del que toma agua',
 'La peruana vive al lado de la casa azul']

## Modelado y Verificación precisa de condiciones en Python

### Clases Auxiliares

In [4]:
from typing import TypedDict, NotRequired
from dataclasses import dataclass
import pandas as pd 
import sys

class Casa(TypedDict):
    nacionalidad: str
    color: str
    serie_que_ve: str
    bebida_que_toma: str
    mascote: str


class CondicionCasa(TypedDict):
    nacionalidad: NotRequired[str]
    color: NotRequired[str]
    serie_que_ve: NotRequired[str]
    bebida_que_toma: NotRequired[str]
    mascote: NotRequired[str]


@dataclass
class NumCasas:
    n: int

    def verificar(self, posible_solucion: list[Casa]) -> bool:
        return len(posible_solucion) == self.n


def _verificar_una(condicion: CondicionCasa, miembro_solucion: Casa) -> bool:
    for condicion_llave, condicion_valor in condicion.items():
        if miembro_solucion[condicion_llave].lower()[:3] != condicion_valor.lower()[:3]:
            return False
    return True


@dataclass
class Miembro:
    condicion: CondicionCasa

    def verificar(self, posible_solucion: list[Casa]) -> bool:
        return any(
            _verificar_una(condicion=self.condicion, miembro_solucion=miembro)
            for miembro in posible_solucion
        )

@dataclass
class MiembroEnPosicion:
    condicion: CondicionCasa
    posicion: int

    def verificar(self, posible_solucion: list[Casa]) -> bool:
        return _verificar_una(
            miembro_solucion=posible_solucion[self.posicion],
            condicion=self.condicion,
        )

@dataclass
class AlLadoDe:
    cond1: CondicionCasa
    cond2: CondicionCasa

    def verificar(self, posible_solucion: list[Casa]) -> bool:
        
        cond1_izq_cond2 = [ 
            _verificar_una(condicion=self.cond1, miembro_solucion=posible_solucion[i]) 
            and 
            _verificar_una(condicion=self.cond2, miembro_solucion=posible_solucion[i+1])
            for i in range(0, len(posible_solucion) - 1)
        ]

        cond2_izq_cond1 = [ 
            _verificar_una(condicion=self.cond2, miembro_solucion=posible_solucion[i]) 
            and 
            _verificar_una(condicion=self.cond1, miembro_solucion=posible_solucion[i + 1])
            for i in range(0, len(posible_solucion) - 1)
        ]

        return any(cond1_izq_cond2) or any(cond2_izq_cond1)

        

### Condiciones codificadas 

In [5]:
CONDICIONES = [
    # 0. En una calle hay 5 casas
    NumCasas(5), 
    # 1. El brasilero vive en la casa roja.
    Miembro({"nacionalidad": "brasilero", "color": "roja"}),
    # 2. El venezolano tiene un perro como mascota.
    Miembro({"nacionalidad": "venezolano", "mascota": "perro"}),
    # 3. El ecuatoriano toma té.
    Miembro({"nacionalidad": "ecuatoriano", "bebida_que_toma": "té"}),
    # 4. La peruana vive en la primera casa.
    MiembroEnPosicion({"nacionalidad": "peruana"}, posicion=0),
    # 5. La colombiana ve "Westworld".
    Miembro({"nacionalidad": "colombiana", "serie_que_ve": "Westworld"}),
    # 6. La casa verde está inmediatamente a la izquierda de la blanca
    AlLadoDe({"color": "verde"}, {"color": "blanca"}),
    # 7. El dueño de la casa verde bebe café.
    Miembro({"color": "verde", "bebida_que_toma": "café"}), 
    # 8. El propietario que ve "The Boys" cría pájaros.
    Miembro({"serie_que_ve": "The Boys", "mascota": "pájaros"}),
    # 9. El dueño de la casa amarilla ve "Dexter".
    Miembro({"color": "amarilla", "serie_que_ve": "Dexter"}),   
    # 10. El hombre que vive en la casa del centro bebe leche.
    MiembroEnPosicion({"bebida_que_toma": "leche"}, posicion=2),
    # 11. El vecino que ve "El Patrón del Mal" vive al lado del que tiene un gato.
    AlLadoDe({"serie_que_ve": "El Patrón del Mal"}, {"mascota": "gato"}),
    # 12. El hombre que tiene un caballo vive al lado del que ve "Dexter".
    AlLadoDe({"mascota": "caballo"}, {"mascota": "gato"}),
    # 13. El propietario que ve "Breaking Bad" toma cerveza.
    Miembro({"bebida_que_toma": "cerveza", "serie_que_ve": "Breaking Bad"}),
    # 14. El vecino que ve "El Patrón del Mal" vive al lado del que toma agua.
    AlLadoDe({"serie_que_ve": "El Patrón del Mal"}, {"bebida_que_toma": "agua"}),
    # 15. La peruana vive al lado de la casa azul.
    AlLadoDe({"nacionalidad": "peruana"}, {"color": "azul"})
]        

### Función maestra de verifiación y reporte de fallos

In [6]:
def verificar_condiciones(posible_solucion: list[Casa]):

    rows = []
    
    for i, (condicion, espanhol) in enumerate(zip(CONDICIONES, CONDICIONES_ESPANHOL)):
        ok = condicion.verificar(posible_solucion)
        # print(f"condicion #{i}: {condicion}\n\tRESULTADO: {'OK' if ok else 'FALLO'}\n")    
        rows.append({
          "indice": i,
          # "condicion": condicion,
          "en espanhol": espanhol,
          "se cumple": ok
        })

    ret = pd.DataFrame(rows)
    num_fallos = (~ret["se cumple"]).sum()

    if num_fallos > 0:
        print(f"\nSolución INVÁLIDA: Número total de fallos: {num_fallos}"
                "\nLas siguientes no se cumplen:\n", file=sys.stderr)
        no_se_cumplen = ret["en espanhol"][~ret["se cumple"]]
        for i, condicion in no_se_cumplen.items():
            print(f"{i}. {condicion}", file=sys.stderr)
        print("\nPor favor intenta nuevamente.", file=sys.stderr)
    else:
        print(f"\nSolución VÁLIDA!!!")
   
    return ret

## Experimentos Modelos de Base (i.e. Propósito general, no de razonamiento) 

### ChatGPT (gratuito)

In [7]:
posible_solucion_chatgpt_1 = [
  {
    "nacionalidad": "peruana",
    "bebida_que_toma": "agua",
    "serie_que_ve": "Dexter",
    "color": "amarilla",
    "mascota": "gato"
  },
  {
    "nacionalidad": "colombiana",
    "bebida_que_toma": "té",
    "serie_que_ve": "Westworld",
    "color": "azul",
    "mascota": "caballo"
  },
  {
    "nacionalidad": "ecuatoriano",
    "bebida_que_toma": "leche",
    "serie_que_ve": "El Patrón del Mal",
    "color": "roja",
    "mascota": "pez"
  },
  {
    "nacionalidad": "venezolano",
    "bebida_que_toma": "cerveza",
    "serie_que_ve": "Breaking Bad",
    "color": "verde",
    "mascota": "perro"
  },
  {
    "nacionalidad": "brasilero",
    "bebida_que_toma": "café",
    "serie_que_ve": "The Boys",
    "color": "blanca",
    "mascota": "pájaro"
  }
]

pd.DataFrame(posible_solucion_chatgpt_1)

,nacionalidad,bebida_que_toma,serie_que_ve,color,mascota
0,peruana,agua,Dexter,amarilla,gato
1,colombiana,té,Westworld,azul,caballo
2,ecuatoriano,leche,El Patrón del Mal,roja,pez
3,venezolano,cerveza,Breaking Bad,verde,perro
4,brasilero,café,The Boys,blanca,pájaro


In [8]:
verificar_condiciones(posible_solucion_chatgpt_1)


Solución INVÁLIDA: Número total de fallos: 5
Las siguientes no se cumplen:

1. El brasilero vive en la casa roja
3. El ecuatoriano toma té
7. El dueño de la casa verde bebe café
11. El vecino que ve "El Patrón del Mal" vive al lado del que tiene un gato
14. El vecino que ve "El Patrón del Mal" vive al lado del que toma agua

Por favor intenta nuevamente.


,indice,en espanhol,se cumple
0,0,En una calle hay cinco casas,True
1,1,El brasilero vive en la casa roja,False
2,2,El venezolano tiene un perro como mascota,True
3,3,El ecuatoriano toma té,False
4,4,La peruana vive en la primera casa,True
5,5,"La colombiana ve ""Westworld""",True
6,6,La casa verde está inmediatamente a la izquier...,True
7,7,El dueño de la casa verde bebe café,False
8,8,"El propietario que ve ""The Boys"" cría pájaros",True
9,9,"El dueño de la casa amarilla ve ""Dexter""",True


In [10]:
posible_solucion_chatgpt_2 = [
  {
    "nacionalidad": "peruana",
    "bebida_que_toma": "agua",
    "serie_que_ve": "Dexter",
    "color": "amarilla",
    "mascota": "gato"
  },
  {
    "nacionalidad": "colombiana",
    "bebida_que_toma": "té",
    "serie_que_ve": "Westworld",
    "color": "azul",
    "mascota": "caballo"
  },
  {
    "nacionalidad": "ecuatoriano",
    "bebida_que_toma": "leche",
    "serie_que_ve": "El Patrón del Mal",
    "color": "roja",
    "mascota": "pez"
  },
  {
    "nacionalidad": "venezolano",
    "bebida_que_toma": "cerveza",
    "serie_que_ve": "Breaking Bad",
    "color": "verde",
    "mascota": "perro"
  },
  {
    "nacionalidad": "brasilero",
    "bebida_que_toma": "café",
    "serie_que_ve": "The Boys",
    "color": "blanca",
    "mascota": "pájaro"
  }
]

pd.DataFrame(posible_solucion_chatgpt_2)

,nacionalidad,bebida_que_toma,serie_que_ve,color,mascota
0,peruana,agua,Dexter,amarilla,gato
1,colombiana,té,Westworld,azul,caballo
2,ecuatoriano,leche,El Patrón del Mal,roja,pez
3,venezolano,cerveza,Breaking Bad,verde,perro
4,brasilero,café,The Boys,blanca,pájaro


In [11]:
verificar_condiciones(posible_solucion_chatgpt_2)


Solución INVÁLIDA: Número total de fallos: 5
Las siguientes no se cumplen:

1. El brasilero vive en la casa roja
3. El ecuatoriano toma té
7. El dueño de la casa verde bebe café
11. El vecino que ve "El Patrón del Mal" vive al lado del que tiene un gato
14. El vecino que ve "El Patrón del Mal" vive al lado del que toma agua

Por favor intenta nuevamente.


,indice,en espanhol,se cumple
0,0,En una calle hay cinco casas,True
1,1,El brasilero vive en la casa roja,False
2,2,El venezolano tiene un perro como mascota,True
3,3,El ecuatoriano toma té,False
4,4,La peruana vive en la primera casa,True
5,5,"La colombiana ve ""Westworld""",True
6,6,La casa verde está inmediatamente a la izquier...,True
7,7,El dueño de la casa verde bebe café,False
8,8,"El propietario que ve ""The Boys"" cría pájaros",True
9,9,"El dueño de la casa amarilla ve ""Dexter""",True


### Chat GPT: modo "deep research"  (agradecimiento a Pablo M.)

In [12]:
chatgpt_deep_research = [
  {
    "nacionalidad": "peruana",
    "bebida_que_toma": "agua",
    "serie_que_ve": "Dexter",
    "color": "amarilla",
    "mascota": "gato"
  },
  {
    "nacionalidad": "colombiana",
    "bebida_que_toma": "té",
    "serie_que_ve": "Westworld",
    "color": "azul",
    "mascota": "caballo"
  },
  {
    "nacionalidad": "ecuatoriano",
    "bebida_que_toma": "leche",
    "serie_que_ve": "El Patrón del Mal",
    "color": "roja",
    "mascota": "perro"
  },
  {
    "nacionalidad": "brasilero",
    "bebida_que_toma": "café",
    "serie_que_ve": "The Boys",
    "color": "verde",
    "mascota": "pájaro"
  },
  {
    "nacionalidad": "venezolano",
    "bebida_que_toma": "cerveza",
    "serie_que_ve": "Breaking Bad",
    "color": "blanca",
    "mascota": "pez"
  }
]


In [13]:
verificar_condiciones(chatgpt_deep_research)


Solución INVÁLIDA: Número total de fallos: 5
Las siguientes no se cumplen:

1. El brasilero vive en la casa roja
2. El venezolano tiene un perro como mascota
3. El ecuatoriano toma té
11. El vecino que ve "El Patrón del Mal" vive al lado del que tiene un gato
14. El vecino que ve "El Patrón del Mal" vive al lado del que toma agua

Por favor intenta nuevamente.


,indice,en espanhol,se cumple
0,0,En una calle hay cinco casas,True
1,1,El brasilero vive en la casa roja,False
2,2,El venezolano tiene un perro como mascota,False
3,3,El ecuatoriano toma té,False
4,4,La peruana vive en la primera casa,True
5,5,"La colombiana ve ""Westworld""",True
6,6,La casa verde está inmediatamente a la izquier...,True
7,7,El dueño de la casa verde bebe café,True
8,8,"El propietario que ve ""The Boys"" cría pájaros",True
9,9,"El dueño de la casa amarilla ve ""Dexter""",True


### Claude 3.5

https://claude.ai/share/e2b9760b-99bc-4543-8a24-2c150c853d8f

In [14]:
# Claude 3.5
posible_solucion_claude_35_1 = [
  {
    "nacionalidad": "peruana",
    "bebida_que_toma": "agua",
    "serie_que_ve": "El Patrón del Mal",
    "color": "amarilla",
    "mascota": "gato"
  },
  {
    "nacionalidad": "ecuatoriana",
    "bebida_que_toma": "té",
    "serie_que_ve": "Dexter",
    "color": "azul",
    "mascota": "caballo"
  },
  {
    "nacionalidad": "brasilera",
    "bebida_que_toma": "leche",
    "serie_que_ve": "The Boys",
    "color": "roja",
    "mascota": "pájaros"
  },
  {
    "nacionalidad": "venezolana",
    "bebida_que_toma": "café",
    "serie_que_ve": "Breaking Bad",
    "color": "verde",
    "mascota": "perro"
  },
  {
    "nacionalidad": "colombiana",
    "bebida_que_toma": "cerveza",
    "serie_que_ve": "Westworld",
    "color": "blanca",
    "mascota": "pez"
  }
]

pd.DataFrame(posible_solucion_claude_35_1) 


,nacionalidad,bebida_que_toma,serie_que_ve,color,mascota
0,peruana,agua,El Patrón del Mal,amarilla,gato
1,ecuatoriana,té,Dexter,azul,caballo
2,brasilera,leche,The Boys,roja,pájaros
3,venezolana,café,Breaking Bad,verde,perro
4,colombiana,cerveza,Westworld,blanca,pez


In [15]:
verificar_condiciones(posible_solucion_claude_35_1)


Solución INVÁLIDA: Número total de fallos: 4
Las siguientes no se cumplen:

9. El dueño de la casa amarilla ve "Dexter"
11. El vecino que ve "El Patrón del Mal" vive al lado del que tiene un gato
13. El propietario que ve "Breaking Bad" toma cerveza
14. El vecino que ve "El Patrón del Mal" vive al lado del que toma agua

Por favor intenta nuevamente.


,indice,en espanhol,se cumple
0,0,En una calle hay cinco casas,True
1,1,El brasilero vive en la casa roja,True
2,2,El venezolano tiene un perro como mascota,True
3,3,El ecuatoriano toma té,True
4,4,La peruana vive en la primera casa,True
5,5,"La colombiana ve ""Westworld""",True
6,6,La casa verde está inmediatamente a la izquier...,True
7,7,El dueño de la casa verde bebe café,True
8,8,"El propietario que ve ""The Boys"" cría pájaros",True
9,9,"El dueño de la casa amarilla ve ""Dexter""",False


In [16]:
# Claude posible solución v.3. 

posible_solucion_claude_35_v3 = [
  {
    "nacionalidad": "peruana",
    "bebida_que_toma": "agua",
    "serie_que_ve": "El Patrón del Mal",
    "color": "roja",
    "mascota": "pez"
  },
  {
    "nacionalidad": "ecuatoriana",
    "bebida_que_toma": "té",
    "serie_que_ve": "The Boys",
    "color": "azul",
    "mascota": "pájaros"
  },
  {
    "nacionalidad": "brasilera",
    "bebida_que_toma": "leche",
    "serie_que_ve": "Dexter",
    "color": "amarilla",
    "mascota": "caballo"
  },
  {
    "nacionalidad": "colombiana",
    "bebida_que_toma": "café",
    "serie_que_ve": "Westworld",
    "color": "verde",
    "mascota": "gato"
  },
  {
    "nacionalidad": "venezolana",
    "bebida_que_toma": "cerveza",
    "serie_que_ve": "Breaking Bad",
    "color": "blanca",
    "mascota": "perro"
  }
]

pd.DataFrame(posible_solucion_claude_35_v3)

,nacionalidad,bebida_que_toma,serie_que_ve,color,mascota
0,peruana,agua,El Patrón del Mal,roja,pez
1,ecuatoriana,té,The Boys,azul,pájaros
2,brasilera,leche,Dexter,amarilla,caballo
3,colombiana,café,Westworld,verde,gato
4,venezolana,cerveza,Breaking Bad,blanca,perro


In [17]:
verificar_condiciones(posible_solucion_claude_35_v3)


Solución INVÁLIDA: Número total de fallos: 3
Las siguientes no se cumplen:

1. El brasilero vive en la casa roja
11. El vecino que ve "El Patrón del Mal" vive al lado del que tiene un gato
14. El vecino que ve "El Patrón del Mal" vive al lado del que toma agua

Por favor intenta nuevamente.


,indice,en espanhol,se cumple
0,0,En una calle hay cinco casas,True
1,1,El brasilero vive en la casa roja,False
2,2,El venezolano tiene un perro como mascota,True
3,3,El ecuatoriano toma té,True
4,4,La peruana vive en la primera casa,True
5,5,"La colombiana ve ""Westworld""",True
6,6,La casa verde está inmediatamente a la izquier...,True
7,7,El dueño de la casa verde bebe café,True
8,8,"El propietario que ve ""The Boys"" cría pájaros",True
9,9,"El dueño de la casa amarilla ve ""Dexter""",True


In [18]:
posible_solucion_claude_35_v5 = [
  {
    "nacionalidad": "peruana",
    "bebida_que_toma": "agua",
    "serie_que_ve": "Breaking Bad",
    "color": "amarilla",
    "mascota": "caballo"
  },
  {
    "nacionalidad": "colombiana",
    "bebida_que_toma": "cerveza",
    "serie_que_ve": "El Patrón del Mal",
    "color": "azul",
    "mascota": "pez"
  },
  {
    "nacionalidad": "brasilera",
    "bebida_que_toma": "leche",
    "serie_que_ve": "Dexter",
    "color": "roja",
    "mascota": "gato"
  },
  {
    "nacionalidad": "ecuatoriana",
    "bebida_que_toma": "té",
    "serie_que_ve": "The Boys",
    "color": "verde",
    "mascota": "pájaros"
  },
  {
    "nacionalidad": "venezolana",
    "bebida_que_toma": "café",
    "serie_que_ve": "Westworld",
    "color": "blanca",
    "mascota": "perro"
  }
]

pd.DataFrame(posible_solucion_claude_35_v5) 


,nacionalidad,bebida_que_toma,serie_que_ve,color,mascota
0,peruana,agua,Breaking Bad,amarilla,caballo
1,colombiana,cerveza,El Patrón del Mal,azul,pez
2,brasilera,leche,Dexter,roja,gato
3,ecuatoriana,té,The Boys,verde,pájaros
4,venezolana,café,Westworld,blanca,perro


In [19]:
verificar_condiciones(posible_solucion_claude_35_v5)


Solución INVÁLIDA: Número total de fallos: 5
Las siguientes no se cumplen:

5. La colombiana ve "Westworld"
7. El dueño de la casa verde bebe café
9. El dueño de la casa amarilla ve "Dexter"
12. El hombre que tiene un caballo vive al lado del que ve "Dexter"
13. El propietario que ve "Breaking Bad" toma cerveza

Por favor intenta nuevamente.


,indice,en espanhol,se cumple
0,0,En una calle hay cinco casas,True
1,1,El brasilero vive en la casa roja,True
2,2,El venezolano tiene un perro como mascota,True
3,3,El ecuatoriano toma té,True
4,4,La peruana vive en la primera casa,True
5,5,"La colombiana ve ""Westworld""",False
6,6,La casa verde está inmediatamente a la izquier...,True
7,7,El dueño de la casa verde bebe café,False
8,8,"El propietario que ve ""The Boys"" cría pájaros",True
9,9,"El dueño de la casa amarilla ve ""Dexter""",False


## Modelos de Razonamiento

### Claude 4 (Sonnet)

https://claude.ai/public/artifacts/a729f300-9e82-48d8-9bdc-87576e77285f

In [20]:
claude_4_sonnet = [
  {
    "nacionalidad": "peruana",
    "bebida_que_toma": "agua",
    "serie_que_ve": "El Patrón del Mal",
    "color": "amarilla",
    "mascota": "gato"
  },
  {
    "nacionalidad": "ecuatoriana",
    "bebida_que_toma": "té",
    "serie_que_ve": "Dexter",
    "color": "azul",
    "mascota": "caballo"
  },
  {
    "nacionalidad": "brasilera",
    "bebida_que_toma": "leche",
    "serie_que_ve": "The Boys",
    "color": "roja",
    "mascota": "pájaros"
  },
  {
    "nacionalidad": "venezolana",
    "bebida_que_toma": "café",
    "serie_que_ve": "Breaking Bad",
    "color": "verde",
    "mascota": "perro"
  },
  {
    "nacionalidad": "colombiana",
    "bebida_que_toma": "cerveza",
    "serie_que_ve": "Westworld",
    "color": "blanca",
    "mascota": "pez"
  }
]

pd.DataFrame(claude_4_sonnet)

,nacionalidad,bebida_que_toma,serie_que_ve,color,mascota
0,peruana,agua,El Patrón del Mal,amarilla,gato
1,ecuatoriana,té,Dexter,azul,caballo
2,brasilera,leche,The Boys,roja,pájaros
3,venezolana,café,Breaking Bad,verde,perro
4,colombiana,cerveza,Westworld,blanca,pez


In [21]:
verificar_condiciones(claude_4_sonnet)


Solución INVÁLIDA: Número total de fallos: 4
Las siguientes no se cumplen:

9. El dueño de la casa amarilla ve "Dexter"
11. El vecino que ve "El Patrón del Mal" vive al lado del que tiene un gato
13. El propietario que ve "Breaking Bad" toma cerveza
14. El vecino que ve "El Patrón del Mal" vive al lado del que toma agua

Por favor intenta nuevamente.


,indice,en espanhol,se cumple
0,0,En una calle hay cinco casas,True
1,1,El brasilero vive en la casa roja,True
2,2,El venezolano tiene un perro como mascota,True
3,3,El ecuatoriano toma té,True
4,4,La peruana vive en la primera casa,True
5,5,"La colombiana ve ""Westworld""",True
6,6,La casa verde está inmediatamente a la izquier...,True
7,7,El dueño de la casa verde bebe café,True
8,8,"El propietario que ve ""The Boys"" cría pájaros",True
9,9,"El dueño de la casa amarilla ve ""Dexter""",False


### Chat-GPT o3 (razonamiento avanzado) (agradecimiento a Darío Muñoz Prudant)

*Darío:*  "Se está rompiendo la cabeza"

*Darío: *  "Se demoró harto _sic_, un par de minutos en resolver"

*Mateo: * "Y sabes cuantos tokens usó en la cadena de razonamiento?"

*Darío: * "muchos najajajaa, varios minutos"



In [22]:
chatgpt_o3 = [
  {
      
    "nacionalidad": "peruana",
    "bebida_que_toma": "agua",
    "serie_que_ve": "Dexter",
    "color": "amarillo",
    "mascota": "gato"
  },
  {
    "nacionalidad": "ecuatoriana",
    "bebida_que_toma": "té",
    "serie_que_ve": "El Patrón del Mal",
    "color": "azul",
    "mascota": "caballo"
  },
  {
    "nacionalidad": "brasileña",
    "bebida_que_toma": "leche",
    "serie_que_ve": "The Boys",
    "color": "rojo",
    "mascota": "pájaros"
  },
  {
    "nacionalidad": "colombiana",
    "bebida_que_toma": "café",
    "serie_que_ve": "Westworld",
    "color": "verde",
    "mascota": "peces"
  },
  {
    "nacionalidad": "venezolana",
    "bebida_que_toma": "cerveza",
    "serie_que_ve": "Breaking Bad",
    "color": "blanco",
    "mascota": "perro"
  }
]

pd.DataFrame(chatgpt_o3)

,nacionalidad,bebida_que_toma,serie_que_ve,color,mascota
0,peruana,agua,Dexter,amarillo,gato
1,ecuatoriana,té,El Patrón del Mal,azul,caballo
2,brasileña,leche,The Boys,rojo,pájaros
3,colombiana,café,Westworld,verde,peces
4,venezolana,cerveza,Breaking Bad,blanco,perro


In [23]:
verificar_condiciones(chatgpt_o3)


Solución VÁLIDA!!!


,indice,en espanhol,se cumple
0,0,En una calle hay cinco casas,True
1,1,El brasilero vive en la casa roja,True
2,2,El venezolano tiene un perro como mascota,True
3,3,El ecuatoriano toma té,True
4,4,La peruana vive en la primera casa,True
5,5,"La colombiana ve ""Westworld""",True
6,6,La casa verde está inmediatamente a la izquier...,True
7,7,El dueño de la casa verde bebe café,True
8,8,"El propietario que ve ""The Boys"" cría pájaros",True
9,9,"El dueño de la casa amarilla ve ""Dexter""",True


In [24]:
cadena_de_razonamiento = """
Avanzando con el rompecabezas
El usuario quiere ayuda con un rompecabezas clásico como el de Zebra, que involucra cinco casas y atributos. Probablemente quiera solo la respuesta en formato JSON. Para resolverlo, podríamos generar todas las permutaciones y luego aplicar las restricciones del rompecabezas. Debería ser posible hacerlo sin necesidad de navegar. Como es estático, podemos procesarlo programáticamente. Creo que lo que necesitan es solo la respuesta final en formato JSON, sin explicaciones detalladas.

El usuario pide la respuesta en formato JSON, así que es importante cumplir con esa instrucción. Sin embargo, creo que es buena idea ofrecer una breve explicación antes de presentar el JSON. De todas formas, soy consciente de que quieren solo JSON, y es probable que eso sea lo que esperan

El usuario quiere la respuesta en formato JSON, pero también podría beneficiarse de una breve explicación. Para ser claro, creo que

El usuario ha pedido la

El usuario ha solicitado específicamente la respuesta en formato JSON, como una lista de diccionarios. Aunque generalmente explic

El usuario ha solicitado específicamente la respuesta en formato JSON, como una lista de diccionarios. Aunque generalmente explicaría el proceso antes, dadas las instrucciones claras de la solicitud, puedo ofrecer una breve explicación antes de presentar solo el código JSON. Creo que eso será un buen enfoque para cubrir ambos aspectos sin desviarme de lo que tienen en mente. Ahora me pongo a resolver

El usuario ha solicitado específicamente la respuesta en formato JSON, como una lista de diccionarios. Aunque generalmente explicaría el proceso antes, dadas las instrucciones claras de la solicitud, puedo ofrecer una breve explicación antes de presentar solo el código JSON. Creo que eso será un buen enfoque para cubrir ambos aspectos sin desviarme de lo que tienen en mente. Ahora me pongo a resolver el rompecabezas y te lo mostraré en el formato requerido.

Puedo intentar resolver este rompecabezas sin usar código, solo aplicando lógica. Sabemos que hay cinco casas y una serie de pistas para determinar las características de cada una: nacionalidades, colores, bebidas, series y mascotas. Por ejemplo, la peruana está en la primera casa, un hombre en la casa del medio bebe leche, la casa verde está a la izquierda de la blanca, y quién bebe café está en la casa verde. Vamos a hacer una tabla y deducir las relaciones entre todos estos atributos.

Parece que el rompecabezas plantea que cada dueño tenga una mascota única, lo que sugiere que hay un pez (peces). Con las pistas dadas, como que el brasileño vive en la casa roja, que la casa amarilla ve Dexter y que el dueño que ve El Patrón del Mal vive junto a los de la mascota gato y el que bebe agua, podemos deducir qué casa corresponde a cada persona, color, mascota, bebida y serie. Ya hemos comenzado a llenar la tabla para resolverlo paso a paso.
Parece que estamos resolviendo cómo asignar colores y nacionalidades a las casas, considerando las restricciones del rompecabezas. Hemos establecido que la casa 3 es roja y pertenece al brasileño, mientras que la casa 1 es amarilla y ve Dexter. Además, la casa 2 tiene una mascota caballo y la casa 4 es verde y toma café.

Ahora tenemos que abordar las mascotas, las bebidas y las series. La clave es encajar cada elemento en su lugar, y con las pistas adicionales, podremos completar la tabla.
En esta parte, intento evaluar las posibles configuraciones para cada casa. Sabemos que la casa 2 es azul y tiene un caballo; la nacionalidad está indefinida, pero sabemos de las restricciones de bebidas y series asociadas. La Casa 1 es peruana y amarilla, ve Dexter y tiene un gato, un perro, pájaros o tal vez un pez. Por lo tanto, descartamos que sea un perro porque solo el venezolano tiene uno. Así que seguimos eliminando y organizando las opciones.

Puedo ver que hay que explorar las posibles conexiones entre las casas. El vecino que ve Dexter está en la casa 1, lo que coloca un caballo en la casa

Estamos trabajando en resolver el rompecabezas, especialmente los vínculos entre hogares y mascotas. Sabemos que la casa 2 tiene un caballo y la casa 3 es desconocida, lo que la convierte en un candidato para el propietario de El Patrón del Mal. Además, aquellos que ven esta serie deben tener vecinos con un gato y agua, lo que limita posibles combinaciones. Ahora, mi tarea es deducir quién posee qué según las pistas, como la vinculación entre las casas y las bebidas.

Parece que tenemos que distribuir los elementos entre varias casas, comenzando con los animales. En la Casa 2, sabemos que hay un caballo, lo que elimina la posibilidad de un gato allí. El gato podría estar en la Casa 4, y el agua podría estar en la Casa 2. Entonces, asignamos el agua en la Casa 2 y el gato en la Casa 4. Respecto a las bebidas, asignamos la leche en la Casa 3 y el café en la Casa 4. Esto nos deja el té, la cerveza y el agua para las casas restantes.
"""

Salida anterior aprox. 1000 tokens

https://platform.openai.com/tokenizer

A costo de hoy esto equivale a $0.04 USD de Salida de o3 ($40.0 / 1M tokens de output)
https://openai.com/api/pricing/


### Gemini 2.5 Flash

https://gemini.google.com/app/1fe81f667a0faddd

In [25]:
posible_solucion_gemini_25_flash = [
  {
    "nacionalidad": "Peruana",
    "bebida_que_toma": "Agua",
    "serie_que_ve": "El Patrón del Mal",
    "color": "Amarilla",
    "mascota": "Caballo"
  },
  {
    "nacionalidad": "Colombiana",
    "bebida_que_toma": "Té",
    "serie_que_ve": "Westworld",
    "color": "Azul",
    "mascota": "Gato"
  },
  {
    "nacionalidad": "Venezolano",
    "bebida_que_toma": "Leche",
    "serie_que_ve": "Breaking Bad",
    "color": "Roja",
    "mascota": "Perro"
  },
  {
    "nacionalidad": "Ecuatoriano",
    "bebida_que_toma": "Cerveza",
    "serie_que_ve": "The Boys",
    "color": "Verde",
    "mascota": "Pájaros"
  },
  {
    "nacionalidad": "Brasilero",
    "bebida_que_toma": "Café",
    "serie_que_ve": "Dexter",
    "color": "Blanca",
    "mascota": "Pez"
  }
]
    

pd.DataFrame(posible_solucion_gemini_25_flash)


,nacionalidad,bebida_que_toma,serie_que_ve,color,mascota
0,Peruana,Agua,El Patrón del Mal,Amarilla,Caballo
1,Colombiana,Té,Westworld,Azul,Gato
2,Venezolano,Leche,Breaking Bad,Roja,Perro
3,Ecuatoriano,Cerveza,The Boys,Verde,Pájaros
4,Brasilero,Café,Dexter,Blanca,Pez


In [26]:
verificar_condiciones(posible_solucion_gemini_25_flash)


Solución INVÁLIDA: Número total de fallos: 6
Las siguientes no se cumplen:

1. El brasilero vive en la casa roja
3. El ecuatoriano toma té
7. El dueño de la casa verde bebe café
9. El dueño de la casa amarilla ve "Dexter"
13. El propietario que ve "Breaking Bad" toma cerveza
14. El vecino que ve "El Patrón del Mal" vive al lado del que toma agua

Por favor intenta nuevamente.


,indice,en espanhol,se cumple
0,0,En una calle hay cinco casas,True
1,1,El brasilero vive en la casa roja,False
2,2,El venezolano tiene un perro como mascota,True
3,3,El ecuatoriano toma té,False
4,4,La peruana vive en la primera casa,True
5,5,"La colombiana ve ""Westworld""",True
6,6,La casa verde está inmediatamente a la izquier...,True
7,7,El dueño de la casa verde bebe café,False
8,8,"El propietario que ve ""The Boys"" cría pájaros",True
9,9,"El dueño de la casa amarilla ve ""Dexter""",False


### Gemini 2.5 pro

In [27]:
posible_solucion_gemini_25_pro_v1 = [
  {
    "nacionalidad": "Peruana",
    "bebida_que_toma": "Agua",
    "serie_que_ve": "Dexter",
    "color": "Amarilla",
    "mascota": "Gato"
  },
  {
    "nacionalidad": "Ecuatoriano",
    "bebida_que_toma": "Té",
    "serie_que_ve": "El Patrón del Mal",
    "color": "Azul",
    "mascota": "Caballo"
  },
  {
    "nacionalidad": "Brasilero",
    "bebida_que_toma": "Leche",
    "serie_que_ve": "The Boys",
    "color": "Roja",
    "mascota": "Pájaros"
  },
  {
    "nacionalidad": "Colombiana",
    "bebida_que_toma": "Café",
    "serie_que_ve": "Westworld",
    "color": "Verde",
    "mascota": "Peces"
  },
  {
    "nacionalidad": "Venezolano",
    "bebida_que_toma": "Cerveza",
    "serie_que_ve": "Breaking Bad",
    "color": "Blanca",
    "mascota": "Perro"
  }
]

pd.DataFrame(posible_solucion_gemini_25_pro_v1) 


,nacionalidad,bebida_que_toma,serie_que_ve,color,mascota
0,Peruana,Agua,Dexter,Amarilla,Gato
1,Ecuatoriano,Té,El Patrón del Mal,Azul,Caballo
2,Brasilero,Leche,The Boys,Roja,Pájaros
3,Colombiana,Café,Westworld,Verde,Peces
4,Venezolano,Cerveza,Breaking Bad,Blanca,Perro


In [28]:
verificar_condiciones(posible_solucion_gemini_25_pro_v1)


Solución VÁLIDA!!!


,indice,en espanhol,se cumple
0,0,En una calle hay cinco casas,True
1,1,El brasilero vive en la casa roja,True
2,2,El venezolano tiene un perro como mascota,True
3,3,El ecuatoriano toma té,True
4,4,La peruana vive en la primera casa,True
5,5,"La colombiana ve ""Westworld""",True
6,6,La casa verde está inmediatamente a la izquier...,True
7,7,El dueño de la casa verde bebe café,True
8,8,"El propietario que ve ""The Boys"" cría pájaros",True
9,9,"El dueño de la casa amarilla ve ""Dexter""",True


## "Usar la herramienta correcta para el trabajo"

(a.k.a. "Using the right tool for the job").

Prolog = "*Pro*grammation en *log*ique"


```prolog
casas(Hs) :-
    % cada casa en la lista se representa por una tupla
    %      h(nacionalidad, mascota, serie de TV, bebida, color_de_casa)
    length(Hs, 5),                                            %  0
    member(h(brasilero , _,_,_, roja), Hs),                   %  1
    member(h(venezolano, perro, _, _, _), Hs),                %  2
    member(h(equatoriano, _, _, té, _), Hs),                  %  3
    Hs = [h(peruana,_,_,_,_)|_],                              %  4	
    member(h(colombiana,_, westworld,_,_), Hs),               %  5
    next(h(_,_,_,_, verde), h(_,_,_,_, blanca), Hs),          %  6
    member(h(_,_,_, café, verde), Hs),                        %  7
    member(h(_,pájaros, the_boys,_,_), Hs),                   %  8	
    member(h(_,_, dexter,_,amarilla), Hs),                    %  9
    Hs = [_,_,h(_,_,_,leche,_),_,_],                          % 10
    next(h(_,_,patron_del_mal ,_,_), h(_,gato   ,_,_,_), Hs), % 11    
    next(h(_,_,dexter,_,_), h(_,caballo,_,_,_), Hs),          % 12
    member(h(_,_, breaking_bad,cerveza,_), Hs),               % 13
    next(h(_,_,patron_del_mal ,_,_), h(_,_,_,agua, _), Hs),   % 14    
    next(h(peruana,_,_,_,_), h(_,_,_,_,azul), Hs),            % 15	
    member(h(_,pez,_,_,_), Hs).		    %  16. alguno tiene un pez

next(A, B, Ls) :- append(_, [A,B|_], Ls).
next(A, B, Ls) :- append(_, [B,A|_], Ls).
```


https://swish.swi-prolog.org/p/acertijo-einsteins-solucion-completa.swinb

## Algunas Referencias y enlace a Repo


  1. Constraint optimization en [Google ORTools](https://developers.google.com/optimization/cp) (paquete de Python): 
  2. [Z3 Prover](https://github.com/Z3Prover/z3) (paquete de Python de Microsoft)
  3. [clingo](https://potassco.org/clingo/) paquete de Python para "Answer Set Programming"
  4. [UV: The Engineering Secrets Behind Python’s Speed King](https://xebia.com/blog/uv-the-engineering-secrets-behind-pythons-speed-king/#:~:text=SAT%2DSolving%20Dependency%20Resolution&text=UV%20instead%20uses%20a%20Conflict,proves%20whether%20a%20solution%20exists) Artículo sobre los aspectos técnicos de UV; menciona la técnica CDCL para resolución SAT
  5. [Lenguage Prolog en Wikipedia](https://en.wikipedia.org/wiki/Prolog)


![](bit.ly-qr.png "Enlace a Repo")

<h1 text-align="center">¡Gracias!</h1>
